# Parsing Regular Expressions with FsLexYacc

F# counterpart of `Python/ANTLR/02-RegExp-Parser.ipynb`. This grammar preserves the original restricted syntax: letters, union `+`, explicit concatenation `⋅`, postfix `*`, and parentheses. The handwritten parser in Chapter 04–05 additionally accepts implicit concatenation, epsilon and the empty language; those are different frontends with clearly specified input languages.

In [ ]:
// Checks remain active in both F# Interactive and the Jupyter kernel.
let checkNotebook condition =
    if not condition then failwith "Notebook check failed"


In [ ]:
#r "Generated/bin/Debug/net9.0/FsLexYacc.Runtime.dll"
#r "Generated/bin/Debug/net9.0/GrammarExamples.dll"
#load "../Support/Visualization.fsx"
open GrammarExamples
open Visualization
open System.IO

## Generated parser

The alternatives return an algebraic syntax tree. Recursive union and product rules associate to the left; repeated stars are accepted as in the original grammar.

```text
%{
open GrammarExamples
%}
%token <char> LETTER
%token PLUS CONCAT STAR LPAREN RPAREN EOF
%start main
%type <GrammarExamples.RegExp> main
%%
main: expression EOF { $1 }
expression:
  | expression PLUS product { Union ($1, $3) }
  | product { $1 }
product:
  | product CONCAT factor { Product ($1, $3) }
  | factor { $1 }
factor:
  | factor STAR { Repeat $1 }
  | LPAREN expression RPAREN { $2 }
  | LETTER { Letter $1 }
```

```text
{
module GrammarExamples.RegExpLexer
open FSharp.Text.Lexing
open GrammarExamples.RegExpParser
}
rule token = parse
| [' ' '\t' '\r']+ { token lexbuf }
| '\n' { lexbuf.EndPos <- lexbuf.EndPos.NextLine; token lexbuf }
| ['a'-'z' 'A'-'Z'] { LETTER (lexbuf.LexemeChar 0) }
| '+' { PLUS }
| '⋅' { CONCAT }
| '*' { STAR }
| '(' { LPAREN }
| ')' { RPAREN }
| eof { EOF }
| _ { failwithf "Illegal character %A" (LexBuffer<char>.LexemeString lexbuf) }
```

In [ ]:
let ast source = Parse.regexp source
let expression = ast "a+b*⋅(a+b⋅b*)"
printfn "%A" expression
checkNotebook (ast "a⋅b+c" = Union (Product (Letter 'a', Letter 'b'), Letter 'c'))
checkNotebook (ast "a**" = Repeat (Repeat (Letter 'a')))
for invalid in ["ab"; "a+"; "(a"; "ε"; "a#"] do
    let rejected = try ast invalid |> ignore; false with _ -> true
    checkNotebook rejected

## Visualize the result

A total tree traversal exposes each semantic constructor to the F# compiler.

In [ ]:
let rec toTree expression =
    match expression with
    | Letter c -> Leaf (string c)
    | Union (left, right) -> Node ("+", [toTree left; toTree right])
    | Product (left, right) -> Node ("⋅", [toTree left; toTree right])
    | Repeat operand -> Node ("*", [toTree operand])
let dot = expression |> toTree |> treeToDot

In [ ]:
Microsoft.DotNet.Interactive.Kernel.display(Microsoft.DotNet.Interactive.Kernel.HTML(renderDot dot), "text/html") |> ignore

## Reuse the automata algorithms

The grammar’s restricted tree is converted explicitly to the full Chapter 04–05 representation. Only the frontend changes; Thompson construction and membership checking have one implementation.

In [ ]:
#load "../Chapter-04-05/AutomataTypes.fsx"
#load "../Chapter-04-05/RegExpToNfa.fsx"
#load "../Chapter-04-05/NfaToDfa.fsx"
open AutomataTypes

let rec toAutomata expression =
    match expression with
    | GrammarExamples.Letter c -> AutomataTypes.Literal c
    | GrammarExamples.Union (left, right) -> AutomataTypes.Alt (toAutomata left, toAutomata right)
    | GrammarExamples.Product (left, right) -> AutomataTypes.Concat (toAutomata left, toAutomata right)
    | GrammarExamples.Repeat operand -> AutomataTypes.Star (toAutomata operand)

let nfa = toAutomata expression |> RegExpToNfa.regexp2nfa (set ['a'; 'b'])
checkNotebook (NfaToDfa.acceptsNfa nfa "a")
checkNotebook (NfaToDfa.acceptsNfa nfa "bba")
checkNotebook (not (NfaToDfa.acceptsNfa nfa ""))